# AutoML Tabular Workflow Pipelines with Modern Python Dependencies

This notebook demonstrates training classification models on Google Cloud Vertex AI using AutoML Tabular Workflows.
It updates official Google Cloud sample notebooks to modern Python package versions and APIs using `google-cloud-pipeline-components>=2.22.0` (`google_cloud_pipeline_components.v1.automl.tabular`).

### Objectives
1. Configure and launch a customized AutoML Tabular training pipeline on the Bank Marketing dataset with automatic Vertex AI Experiment tracking.
2. Extract the hyperparameter tuning result artifact from Stage 1.
3. Launch a Skip Architecture Search AutoML Tabular pipeline reusing Stage 1 tuning results to save time and cost.
4. List and inspect experiment runs automatically logged in Vertex AI Experiments.

In [1]:
from dotenv import load_dotenv

from tabflows import (
    TabularPipelineConfig,
    create_tabular_pipeline_job,
    list_experiment_runs,
    run_skip_architecture_search_pipeline,
)

# Load environment variables from local .env file
load_dotenv()
print("Environment and libraries loaded successfully.")

Environment and libraries loaded successfully.


In [2]:
# TabularPipelineConfig automatically loads GCP_PROJECT, GCP_LOCATION, and GCP_BUCKET_URI from .env
config = TabularPipelineConfig()

print(f"Project ID: {config.project_id}")
print(f"Location: {config.location}")
print(f"Bucket URI: {config.bucket_uri}")
print(f"Pipeline Root DIR: {config.root_dir}")
print(f"Transform Config Path: {config.transform_config_path}")

Project ID: hybrid-vertex
Location: us-central1
Bucket URI: gs://jts-tabflows-v1
Pipeline Root DIR: gs://jts-tabflows-v1/automl_tabular_pipeline
Transform Config Path: gs://jts-tabflows-v1/automl_tabular_pipeline/transform_config_unique.json


In [3]:
# Build pipeline template and parameter dictionary
job_id = "automl-tabular-run-01"

# create_tabular_pipeline_job constructs the PipelineJob and automatically logs the experiment run
job = create_tabular_pipeline_job(
    config=config,
    job_id=job_id,
    log_experiment=True,
)

# Alternatively, manually log an existing PipelineJob to Vertex AI Experiments:
# log_experiment_run(run_name=job_id, pipeline_job=job, config=config)

print(
    f"PipelineJob object '{job_id}' created and logged to "
    f"Vertex AI Experiment '{config.experiment_name}'."
)
# To execute job on Vertex AI: job.run()

PipelineJob object created successfully: automl-tabular-run-01


## Skip Architecture Search Pipeline & Automatic Experiment Tracking

Reusing the hyperparameter tuning result from the stage-1 tuner task reduces training time and cost.

Extract the tuning result artifact URI from `automl-tabular-stage-1-tuner` and pass it to `run_skip_architecture_search_pipeline`, which automatically tracks the job run in Vertex AI Experiments.

In [4]:
# Example: Extract tuning result artifact URI after job execution:
# pipeline_task_details = job.gca_resource.job_detail.task_details
# stage_1_task = get_task_detail(pipeline_task_details, "automl-tabular-stage-1-tuner")
# stage_1_tuning_result_artifact_uri = (
#     stage_1_task.outputs["tuning_result_output"].artifacts[0].uri
# )

stage_1_tuning_result_artifact_uri = f"{config.root_dir}/tuning_result_artifact"

skip_job_id = "automl-tabular-skip-search-01"

# run_skip_architecture_search_pipeline constructs the skip search PipelineJob
skip_job = run_skip_architecture_search_pipeline(
    config=config,
    tuning_result_artifact_uri=stage_1_tuning_result_artifact_uri,
    job_id=skip_job_id,
    log_experiment=True,
)

print(
    f"Skip Architecture Search PipelineJob '{skip_job_id}' created and tracked in "
    f"experiment '{config.experiment_name}'."
)
# To execute job on Vertex AI: skip_job.run()

Skip Architecture Search PipelineJob object created successfully: automl-tabular-skip-search-01


In [ ]:
# Retrieve and display experiment runs logged in Vertex AI Experiments
print(f"Fetching logged experiment runs for '{config.experiment_name}'...")
try:
    df_runs = list_experiment_runs(config=config)
    if df_runs is not None and not df_runs.empty:
        print(f"Found {len(df_runs)} experiment run(s):")
        print(df_runs.head())
    else:
        print(f"No experiment runs found yet under '{config.experiment_name}'.")
except Exception as e:
    print(f"Notice: Vertex AI Experiment '{config.experiment_name}' status: {e}")